# 3. A building, broken down

This tutorial assesses a whole inventory with
[`assess_building()`](../api/inventory.md#dlca.inventory.assess_building)
and reads its breakdowns by module, gas and material. It then looks at two
kinds of CO₂ taken up from the air: biogenic carbon in timber and
carbonation of concrete.

!!! note "Illustrative numbers"
    The inputs are `timber_biogenic.json` and `concrete.json` from
    [`tests/data/inventories/`](https://github.com/jayarehart/DLCA/tree/main/tests/data/inventories):
    the floor structure of a made-up 1,000 m² office floor. Every number in
    them is made up, so the results are not a comparison of timber and
    concrete.

In [ ]:
import json
from pathlib import Path

# The example inventories live in the repository's tests/data/inventories/.
# Look for them from the folder this notebook runs in, upwards.
here = Path.cwd().resolve()
DATA = next(
    p / "tests" / "data" / "inventories"
    for p in [here, *here.parents]
    if (p / "tests" / "data" / "inventories").is_dir()
)


def load(name):
    """The example file `name`.json as a plain dict."""
    return json.loads((DATA / f"{name}.json").read_text(encoding="utf-8"))

## Assess the timber floor

The timber floor declares the biogenic carbon held in its wood. Its uptake
from the air is spread over the years of forest regrowth after harvest, and
a very small part of it falls in year 100 or later. `assess_building()`
warns about that, because the sliding series count such late emissions as
zero. Here the warning is caught and printed.

In [ ]:
import warnings

from dlca import assess_building
from dlca.inventory import BuildingInventory

timber = BuildingInventory.model_validate(load("timber_biogenic"))
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = assess_building(timber, parameters="AR6", horizon=100)
for w in caught:
    print("Warning:", w.message)
print("series length:", result.years, "years")
print("rows with emissions in year 100 or later:", dict(result.sliding_incomplete))

## By module

`by_module` has one row per EN 15978 module the building gives, plus the
uptake. Each row is a [`Metrics`](../api/inventory.md#dlca.inventory.Metrics)
with six series (GWI and GTP; dynamic, sliding and static).

In [ ]:
import pandas as pd


def table(rows, year=100):
    """GWI and GTP of each row in one year, in kg CO2e."""
    return pd.DataFrame(
        {
            f"GWI dynamic, year {year}": {str(k): m.gwi_dynamic[year] for k, m in rows.items()},
            f"GTP dynamic, year {year}": {str(k): m.gtp_dynamic[year] for k, m in rows.items()},
        }
    ).round(0)


by_module = table(result.by_module)
by_module

The rows add up to the total:

In [ ]:
print("sum of rows:", round(by_module.iloc[:, 0].sum()))
print("total:      ", round(result.total.gwi.dynamic[100]))

## By gas and by material

In [ ]:
table(result.by_gas)

In [ ]:
table(result.by_material)

The `co2` row is negative because it holds the biogenic uptake. The `ch4`
row is non-fossil methane from wood that decays in landfill (module C4).

## The biogenic carbon balance

`result.biogenic` (the same as
[`biogenic_balance()`](../api/inventory.md#dlca.inventory.biogenic_balance))
gives, per material, the carbon declared, released at end of life and still
stored, in kg CO₂.

In [ ]:
pd.DataFrame(
    {
        str(key): {
            "declared": b.declared,
            "uptake": b.uptake,
            "released as CO2": b.released_co2,
            "released as CH4 (as CO2)": b.released_ch4_as_co2,
            "stored": b.stored,
        }
        for key, b in result.biogenic.items()
    }
).round(1)

The uptake is a little less than the declared carbon: the carbon that
leaves as non-fossil methane is already counted in the `ch4` emission, so
it is left out of the uptake.

## Concrete carbonation

The concrete floor has a `carbonation` field on its slab. The package turns
it into two uptake modules: `carbonation-uptake` during use and
`carbonation-eol` after the concrete is crushed at end of life.

In [ ]:
from dlca.inventory import CARBONATION_EOL_MODULE, CARBONATION_UPTAKE_MODULE, timeline

concrete = BuildingInventory.model_validate(load("concrete"))
concrete_result = assess_building(concrete, parameters="AR6", horizon=100)

modules = timeline(concrete, horizon=100).by_module()
for name in (CARBONATION_UPTAKE_MODULE, CARBONATION_EOL_MODULE):
    co2 = modules[name]["co2"]
    years = co2.nonzero()[0]
    when = f"year {years[0]}" if len(years) == 1 else f"years {years[0]}-{years[-1]}"
    print(f"{name}: {co2.sum():,.0f} kg CO2, {when}")

In [ ]:
table(concrete_result.by_module)

Carbonation offsets part of the slab's A1–A3 emissions in this example. How
much concrete really takes up, especially in stockpiles at end of life, is
uncertain; see decision record
[0017](../decisions/0017-concrete-carbonation.md).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, r in {"timber (biogenic)": result, "concrete": concrete_result}.items():
    ax.plot(r.total.gwi.dynamic / 1000, label=name)
ax.axhline(0, color="grey", linewidth=0.5)
ax.set(title="GWI, dynamic (illustrative inputs)", xlabel="year", ylabel="t CO2e")
ax.legend()
fig.tight_layout()

The timber curve rises at first and then falls as the forest regrows and
takes CO₂ back from the air. The shape depends on the made-up rotation and
release amounts in the input.

Next: [scenarios](04_scenarios.ipynb).